# GNN-BERT Music Context Demo

This notebook simulates a 10-second synthetic music clip and uses the project pipeline to:
- create audio segments
- build a PyG graph
- run forward inference through the Task 3 fusion model
- print predicted context tags and valence/arousal values
- visualize the graph adjacency structure

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import torch

ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

if (ROOT / "src").exists():
    sys.path.insert(0, str(ROOT / "src"))

from audio_features import AudioProcessor
from bert_encoder import BERTTextEncoder
from fusion_model import GNNBERTFusionModel
from gnn_model import MusicGNNEncoder
from graph_builder import MusicGraphBuilder

print("Project root:", ROOT)
print("Torch version:", torch.__version__)

In [ ]:
# Generate synthetic 10-second audio clip and text description
sr = 22050
seconds = 10
samples = int(sr * seconds)

t = np.linspace(0, seconds, samples, endpoint=False)
melody = 0.7 * np.sin(2 * np.pi * 220 * t)
chords = 0.4 * np.sin(2 * np.pi * 110 * t)
beat = 0.3 * np.sin(2 * np.pi * 2.5 * t)
noise = 0.05 * np.random.randn(samples)
waveform = (melody + chords + beat + noise).astype(np.float32)

text_prompt = "a warm, mellow instrumental track with soft synth pads, gentle strings, and a calm emotional atmosphere"
text_tags = [
    "calm", "warm", "ambient", "instrumental", "melodic", "soft", "gentle"
]

print(f"Synthetic waveform shape: {waveform.shape}")
print(f"Text prompt: {text_prompt}")

In [ ]:
# Extract audio segments and build graph
processor = AudioProcessor(
    sample_rate=22050,
    n_mels=128,
    n_chroma=12,
    n_mfcc=20,
    segment_window_sec=5.0,
    segment_hop_sec=2.5,
)

segments = processor.segment_signal(waveform)
segment_vectors = [processor.summarize_segment(seg) for seg in segments]

print(f"Number of segments: {len(segments)}")
print(f"Node feature shape: {segment_vectors[0].shape}")

graph_builder = MusicGraphBuilder(similarity_threshold=0.7, temporal_window=3)
graph = graph_builder.build_graph(segment_vectors)
print(f"Graph node count: {graph.x.shape[0]}")
print(f"Graph edge count: {graph.edge_index.shape[1]}")

In [ ]:
# Create tiny encoder and fusion model instances for inference
bert_model = BERTTextEncoder(model_name="distilbert-base-uncased", max_length=32, hidden_dim=768)
text_encoder = bert_model.eval()

gnn_model = MusicGNNEncoder(in_channels=segment_vectors[0].shape[0], hidden_dim=64, num_layers=2, model_type="sage")
fusion_model = GNNBERTFusionModel(graph_dim=64, text_dim=768, fusion_dim=128, num_tags=20)

with torch.no_grad():
    H_text, t = text_encoder([text_prompt])
    g = gnn_model(graph)
    tag_logits, valence_pred, arousal_pred = fusion_model(g, H_text)

print(f"H_text shape: {H_text.shape}")
print(f"g shape: {g.shape}")
print(f"tag_logits shape: {tag_logits.shape}")
print(f"valence_pred shape: {valence_pred.shape}")
print(f"arousal_pred shape: {arousal_pred.shape}")

In [ ]:
# Decode outputs into predicted context tags and continuous emotion values
probs = torch.sigmoid(tag_logits[0])
threshold = 0.5
predicted_tags = [tag for tag, p in zip(text_tags, probs.tolist()) if p >= threshold][:10]

valence_value = float(valence_pred[0, 0].item())
arousal_value = float(arousal_pred[0, 0].item())

print("Predicted context tags:", predicted_tags)
print(f"Predicted valence: {valence_value:.4f}")
print(f"Predicted arousal: {arousal_value:.4f}")

# Optional: normalize/clip values for readability
print(f"Clipped valence: {float(np.clip(valence_value, -1.0, 1.0)):.4f}")
print(f"Clipped arousal: {float(np.clip(arousal_value, -1.0, 1.0)):.4f}")

In [ ]:
# Visualize the adjacency graph
G = nx.Graph()
for i in range(graph.x.size(0)):
    G.add_node(i)

edge_index = graph.edge_index.t().tolist()
for src, dst in edge_index:
    if src != dst:
        G.add_edge(src, dst)

plt.figure(figsize=(8, 6))
pos = nx.spring_layout(G, seed=42)
nx.draw(G, pos, with_labels=True, node_color="lightblue", node_size=700, edge_color="gray")
plt.title("Audio segment adjacency graph")
plt.tight_layout()
plt.show()

## Summary

This demo shows the end-to-end workflow of the project prototype: synthetic audio -> segment features -> graph construction -> GNN embedding -> BERT text encoding -> cross-attention fusion -> tag and emotion prediction.